# 015 - QAOA via QPU 

## 1.00 Admin  

#### 1.10 Libraries

In [1]:
from qiskit_ibm_runtime import (
    Batch,
    SamplerV2,
    QiskitRuntimeService,
)

# for circuit graphics 
import pylatexenc

# for the environment report 
import environment as envir 
from tabulate import tabulate 

# Math 
import numpy as np

# Qiskit libraries 
from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import COBYLA

from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import n_local

from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_finance.applications.optimization import PortfolioOptimization

from qiskit import transpile

#### 1.10 Environment 

In [2]:
data, headers = envir.environment_state()
print(tabulate(data, headers=headers))

System/Library           Version
-----------------------  ---------
Python                   3.11.15
numpy                    2.4.3
torch                    2.10.0
matplotlib               3.10.8
seaborn                  0.13.2
qiskit                   2.2.0
qiskit_machine_learning  0.9.0
qiskit_optimization      0.7.0
qiskit_ibm_runtime       0.45.1
pylatexenc               2.10
qiskit_algorithms        0.4.0
yfinance                 1.2.0
qiskit_aer               0.15.1
qiskit-finance           0.4.1


#### 1.20 Notebook Parameters 

In [3]:
# Set Seed 
algorithm_globals.random_seed = 1234

# Print debug 
print_flag = True 

#### 1.30 Platform Authentication

In [4]:
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    instance="ptc_work", 
)

#### 1.40 Backend Selection (Least Busy) 

In [5]:
# Filter for real devices (exclude simulators) and those that are operational
backends = service.backends(
    simulator=False,
    operational=True
)

# Sort by queue length
least_busy = sorted(backends, key=lambda b: b.status().pending_jobs)[0]

backend = least_busy
print(f"Using least busy backend: {backend.name}")
print(f"Qubits:", backend.num_qubits)
print(f"Pending jobs: {backend.status().pending_jobs}")

Using least busy backend: ibm_fez
Qubits: 156
Pending jobs: 0


### 2.00 Package Build 

#### 2.10 Data  

In [6]:
# ----------------------------------- 
# In this run we are dealing with synthetic data 
# later we can subsitiute real data with the 
# asset_fetch function 
# 
# ---------------------------------------------

# set number of assets - We will move thie later 
num_assets = 4

expected_returns = np.array([0.10, 0.20, 0.15, 0.18])

covariances = np.array([
    [0.005, -0.010, 0.004, -0.002],
    [-0.010, 0.040, -0.002, 0.004],
    [0.004, -0.002, 0.023, 0.002],
    [-0.002, 0.004, 0.002, 0.018],
])

risk_factor = 0.5
budget = 2


#### 2.20 Model 

In [7]:
portfolio = PortfolioOptimization(
    expected_returns,
    covariances,
    risk_factor,
    budget
)

qp = portfolio.to_quadratic_program()

if print_flag: 
    print(qp.prettyprint())

Problem name: Portfolio optimization

Minimize
  0.0025*x_0^2 - 0.01*x_0*x_1 + 0.004*x_0*x_2 - 0.002*x_0*x_3 + 0.02*x_1^2
  - 0.002*x_1*x_2 + 0.004*x_1*x_3 + 0.0115*x_2^2 + 0.002*x_2*x_3 + 0.009*x_3^2
  - 0.1*x_0 - 0.2*x_1 - 0.15*x_2 - 0.18*x_3

Subject to
  Linear constraints (1)
    x_0 + x_1 + x_2 + x_3 == 2  'c0'

  Binary variables (4)
    x_0 x_1 x_2 x_3



#### 2.30 Ansatz

In [8]:
ansatz = n_local(
    num_qubits=num_assets,
    rotation_blocks=["ry"],
    entanglement_blocks="cz",
    entanglement="linear",
    reps=2
)

#### 2.31 Transpile Ansatz

In [9]:
# Transpile the ansatz for the target backend
ansatz = transpile(
    ansatz,
    backend=backend,
    optimization_level=3
)

#### 2.40 Optimizer 

In [10]:
optimizer = COBYLA(maxiter=500)

### 3.00 Job Submission 

#### 3.10 Assign primitive and submit job 

In [ ]:
with Batch(backend=backend) as batch:
    
    sampler = SamplerV2(
        mode=batch,
        options={
            "execution": {
                "shots": 2048
            }
        }
    )

    vqe = SamplingVQE(
        sampler=sampler,
        ansatz=ansatz,
        optimizer=optimizer
    )

    solver = MinimumEigenOptimizer(vqe)

    result = solver.solve(qp)

### 4.0 Post Processing 

In [ ]:
print("Optimal portfolio:", result.x)
print("Optimal value:", result.fval)

In [ ]:
print("Objective value from qp:", qp.objective.evaluate(result.x))